In [1]:
import os
import shutil
import sys
import subprocess

In [2]:
program_path = "../src/GenerateDataset/build_release"

# Dataset Path
dataset_path = "../data/generated_Nov22_20224"

# Clean the dataset directory
if os.path.exists(dataset_path):
    shutil.rmtree(dataset_path)

# Create the dataset directory
if not os.path.exists(dataset_path):
    os.makedirs(dataset_path)

# Create images directory
images_path = dataset_path + "/images"
if not os.path.exists(images_path):
    os.makedirs(images_path)

# Create xml files directory
xml_path = dataset_path + "/xml"
if not os.path.exists(xml_path):
    os.makedirs(xml_path)

output_path = "output"
# Get absolute path
output_path = os.path.abspath(output_path)

In [ ]:
n_iter = 100


# Remove and recreate output folder
shutil.rmtree(output_path, ignore_errors=True)
os.makedirs(output_path, exist_ok=True)

# Generate dataset
for i in range(n_iter):
    print("Generating image: ", i)
    seed = i
    image_name = f"cowpea_{i:04d}" 
    # Generate image 
    # Construct the command
    command = ""
    command += f"cd {program_path} && ./main " 
    command += f"-h 1.0 -o {output_path} -seed {seed} -name {image_name} -xml -tile black -g"
    result = subprocess.run(command, shell=True, capture_output=True, text=True)


# Check if the command was successful
if result.returncode == 0:
    # print("Command executed successfully")
    # print(result.stdout)  # Print the standard output
    pass
else:
    print(result.stdout)  # Print the standard output
    print(result.stderr)  # Print the error output
    raise("Command failed")
    pass

Generating image:  0
Generating image:  1
Generating image:  2
Generating image:  3
Generating image:  4
Generating image:  5
Generating image:  6
Generating image:  7
Generating image:  8
Generating image:  9
Generating image:  10
Generating image:  11
Generating image:  12
Generating image:  13
Generating image:  14
Generating image:  15
Generating image:  16
Generating image:  17
Generating image:  18
Generating image:  19
Generating image:  20
Generating image:  21
Generating image:  22
Generating image:  23
Generating image:  24
Generating image:  25
Generating image:  26
Generating image:  27
Generating image:  28
Generating image:  29
Generating image:  30
Generating image:  31
Generating image:  32
Generating image:  33
Generating image:  34
Generating image:  35
Generating image:  36
Generating image:  37
Generating image:  38
Generating image:  39
Generating image:  40
Generating image:  41
Generating image:  42
Generating image:  43
Generating image:  44
Generating image:  4

In [3]:
import os
import subprocess
import concurrent.futures
from tqdm import tqdm

# Function to re-render a single XML file
def re_render_xml(filename):
    if filename.endswith(".xml"):
        image_name = filename.split(".")[0]
        # Generate image 
        # Construct the command
        command = ""
        command += f"cd {program_path} && ./main " 
        command += f"-h 1.0 -o {output_path} -name {image_name} -tile black -f {os.path.join(output_path, filename)}"
        result = subprocess.run(command, shell=True, capture_output=True, text=True)
        return result

# Re-render XML files in parallel with progress bar
with concurrent.futures.ThreadPoolExecutor() as executor:
    xml_list = os.listdir(output_path)
    xml_list.sort()
    futures = [executor.submit(re_render_xml, filename) for filename in xml_list if filename.endswith(".xml")]
    
    # Use tqdm to show progress bar
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Re-rendering XML files"):
        result = future.result()
        # Check if the command was successful
        if result.returncode != 0:
            print(result.stdout)  # Print the standard output
            print(result.stderr)  # Print the error output
            raise Exception("Command failed")

Re-rendering XML files: 100%|██████████| 2000/2000 [06:28<00:00,  5.15it/s]


In [4]:
# List jpg and xml files and move them to the dataset directory
for filename in os.listdir(output_path):
    if filename.endswith(".jpeg"):
        shutil.move(os.path.join(output_path, filename), images_path)
    elif filename.endswith(".xml"):
        shutil.move(os.path.join(output_path, filename), xml_path)

In [8]:
# Test loading the generated dataset
from plant_dataset import PlantDataset 
# Show some images
import matplotlib.pyplot as plt
import numpy as np
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from plant_dataset import collate_fn
transform = transforms.Compose([
                        transforms.ToTensor(),
                        # transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])
train_dataset = PlantDataset("../data/generated_Nov14_20224", load_depth=False, preload=False, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)
import cv2
n = 5
for i in range(n):
    #image, vecs, _ = train_dataset[-i-1]
    #image, vecs, _ = train_dataset[i]
    image, vecs, _ = next(iter(train_loader))
    image = image.permute(1, 2, 0)
    image_rgb = image[:, :, :3]
    img = cv2.normalize(np.array(image_rgb.cpu()), None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    # img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.subplot(5, 2, 2*i+1)
    plt.imshow(img)
    plt.title(f"Image {i}")
    plt.axis('off')

    if train_dataset.load_depth:
        image_depth = image[:, :, 3]
        img = cv2.normalize(np.array(image_depth.cpu()), None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
        plt.subplot(5, 2, 2*i+2)
        plt.imshow(img, cmap='gray')
        plt.title("Depth")
        plt.axis('off')

Total 300 images and plant strings loaded


AttributeError: 'NoneType' object has no attribute 'shape'